# [5.2] Gemma Scope and Feature Steering - Exercises

Implement the local validation ladder for sparse features: sparsity metrics, reconstruction metrics, direct logit attribution, held-out feature validation, top activations, ablation, and decoder-vector steering controls.

```yaml
gt_tier: GT-3
exercise_id: 5.2-gemma-scope-feature-steering
expected_runtime: 45-75 minutes for CPU exercises; a few minutes for CUDA artifact preflight
requires_gpu: true for the Gemma Scope artifact preflight; false for the implementation exercises
```

Reading map: review Chapter 1 toy superposition/SAE material, then skim Gemma Scope and sparse feature circuit validation methods. Common failure mode: accepting max-activating examples as an interpretation without held-out positives, matched negatives, steering, ablation, and random controls.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part2_gemma_scope_feature_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_feature_steering.tests as tests


@dataclass(frozen=True)
class SAEReconstructionMetrics:
    l0: float
    feature_density_mean: float
    dead_feature_fraction: float
    reconstruction_mse: float
    reconstruction_kl: float | None = None
    loss_recovered: float | None = None


@dataclass(frozen=True)
class FeatureDetectionReport:
    auc: float
    positive_mean: float
    negative_mean: float
    separation: float
    threshold_accuracy: float


@dataclass(frozen=True)
class SteeringComparisonReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    passes_control: bool


## Sparsity Metrics

Difficulty: medium. Importance: high. Expected output: density, L0, and dead-feature tests should pass. Common bug: reducing over the feature dimension when computing per-feature density.


In [ ]:
def feature_density(feature_acts: t.Tensor, threshold: float = 0.0) -> t.Tensor:
    raise NotImplementedError()


def l0(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


def dead_feature_fraction(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


tests.test_feature_density_l0_and_dead_fraction_match_reference(
    feature_density,
    l0,
    dead_feature_fraction,
)


## Reconstruction Metrics

Difficulty: medium. Importance: high. Expected output: the metric function should match manual MSE/loss-recovered checks and the independent reference. Common bug: computing KL in the wrong direction.


In [ ]:
def mean_kl_divergence(reference_logits: t.Tensor, reconstructed_logits: t.Tensor) -> float:
    raise NotImplementedError()


def loss_recovered(
    *,
    clean_loss: float,
    reconstructed_loss: float,
    zero_ablation_loss: float,
) -> float:
    raise NotImplementedError()


def compute_sae_reconstruction_metrics(
    *,
    activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    feature_acts: t.Tensor,
    threshold: float = 0.0,
    reference_logits: t.Tensor | None = None,
    reconstructed_logits: t.Tensor | None = None,
    clean_loss: float | None = None,
    reconstructed_loss: float | None = None,
    zero_ablation_loss: float | None = None,
) -> SAEReconstructionMetrics:
    raise NotImplementedError()


tests.test_compute_sae_reconstruction_metrics_matches_manual_and_reference(
    compute_sae_reconstruction_metrics,
)


## Direct Logit Attribution

Difficulty: medium. Importance: high. Expected output: selected token effects should equal `decoder_vectors @ unembedding`. Common bug: transposing the unembedding because it feels like a projection.


In [ ]:
def direct_logit_attribution(
    decoder_vectors: t.Tensor,
    unembedding: t.Tensor,
    token_ids: t.Tensor | list[int] | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_direct_logit_attribution_projects_decoder_vectors(direct_logit_attribution)


## Held-Out Feature Validation

Difficulty: medium. Importance: high. Expected output: perfect held-out separation should give AUC 1.0, and tied scores should use average ranks. Common bug: evaluating only top activations and never testing matched negatives.


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def feature_detection_report(
    scores: t.Tensor,
    labels: t.Tensor,
    threshold: float | None = None,
) -> FeatureDetectionReport:
    raise NotImplementedError()


tests.test_feature_detection_report_handles_heldout_controls(
    feature_detection_report,
    roc_auc_binary,
)


## Top Activations and Ablation

Difficulty: medium. Importance: high. Expected output: top-k should be computed on the selected feature surface, and zero/mean ablation should match the reference. Common bug: flattening every feature before `topk`.


In [ ]:
def top_activating_examples(
    feature_acts: t.Tensor,
    feature_id: int,
    k: int = 10,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


def ablate_features(
    feature_acts: t.Tensor,
    feature_ids: t.Tensor | list[int],
    replacement: Literal["zero", "mean"] = "zero",
) -> t.Tensor:
    raise NotImplementedError()


tests.test_top_activating_examples_and_ablation_match_reference(
    top_activating_examples,
    ablate_features,
)


## Decoder Steering Controls

Difficulty: medium. Importance: high. Expected output: last-token steering should leave earlier tokens unchanged, all-position steering should broadcast the vector, and feature steering should beat the random-control delta. Common bug: mutating the clean activations in place.


In [ ]:
def apply_decoder_steering(
    activations: t.Tensor,
    decoder_vectors: t.Tensor,
    feature_ids: t.Tensor | list[int],
    coefficients: t.Tensor | list[float] | float,
    *,
    positions: Literal["all", "last"] = "last",
) -> t.Tensor:
    raise NotImplementedError()


def steering_comparison_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
) -> SteeringComparisonReport:
    raise NotImplementedError()


tests.test_decoder_steering_and_random_control_report(
    apply_decoder_steering,
    steering_comparison_report,
)


## Verification Block

After completing the exercises, compare against `solutions.py`. The CUDA report verifies this control ladder on CUDA tensors, loads the pinned Gemma Scope SAE artifact, and validates a semantic feature hypothesis on authenticated real Gemma 3 activations against random-feature and label-shuffle controls.


In [ ]:
# Uncomment after checking your implementations against the section solution.
# from part2_gemma_scope_feature_steering.solutions import run_smoke_test, run_gpu_test
# run_smoke_test(cpu=True)
# run_gpu_test(max_vram_gb=24.0)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
